# Financial QLoRA SFT → DPO pipeline

[Open this notebook in Colab](https://colab.research.google.com/github/Hydaspex/qwen3-financial-sft-unsloth-dpo/blob/feat/colab-pipeline/notebooks/colab_pipeline.ipynb)

This notebook runs the portfolio pipeline on a CUDA GPU: TAT-QA preparation → Unsloth QLoRA SFT → DPO preference optimisation → base/SFT/DPO comparison. It uses a small smoke-run configuration by default.

**British English:** optimisation, modelling, licence and artefact.


## 1. Runtime check

Use **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU). TPU runtimes are detected for clarity but are not supported by this Unsloth CUDA recipe.


In [ ]:
import os
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())
else:
    raise RuntimeError('Enable a CUDA GPU runtime. TPU and CPU runtimes are not supported by this Unsloth recipe.')


## 2. Optional Google Drive storage

Mount Drive if you want to copy adapters or logs out of the temporary Colab filesystem.


In [ ]:
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT = '/content/drive/MyDrive/qwen3-financial-sft-unsloth-dpo'
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)


## 3. Install the branch

This cell is idempotent: it always removes any existing clone and re-clones from the branch, so the notebook can be re-run safely.


In [ ]:
import os
import shutil
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Hydaspex/qwen3-financial-sft-unsloth-dpo.git'
BRANCH = 'feat/colab-pipeline'
REPO_DIR = Path('/content/qwen3-financial-sft-unsloth-dpo')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone -q --branch $BRANCH $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
!pip install -q -e ".[dev]"

# A live kernel's sys.path is only set up from installed .pth/editable-hook
# files at interpreter startup, so a pip install -e run mid-session doesn't
# always make 'finpost' importable without this explicit fallback.
src_dir = str(REPO_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from finpost.config import load_config
print('finpost imported successfully from', Path.cwd())


## 4. Configure a smoke run

The default run uses 200 examples, one epoch and batch size one. Set `MAX_SAMPLES = None` for a full data preparation run, but use a larger GPU and expect a longer runtime.


In [ ]:
from pathlib import Path
import yaml

MAX_SAMPLES = 200
config_path = Path('configs/post_training.yaml')
config = yaml.safe_load(config_path.read_text())
config['data']['max_samples'] = MAX_SAMPLES
config['sft']['epochs'] = 1
config['sft']['batch_size'] = 1
config['dpo']['epochs'] = 1
config['dpo']['batch_size'] = 1

# Absolute sqlite URI: MLflow's default file store resolves './mlruns'
# relative to the current working directory, which breaks after a Colab
# runtime restart (cwd resets to /content unless cell 3 is re-run). An
# absolute path keeps every stage writing to the same store.
MLFLOW_DB = (REPO_DIR / 'mlflow.db').resolve()
config['mlflow']['tracking_uri'] = f'sqlite:////{MLFLOW_DB}'

config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())


## 5. Prepare TAT-QA data


In [ ]:
!python scripts/prepare_data.py --config configs/post_training.yaml


## 6. Unsloth QLoRA SFT


In [ ]:
!python scripts/train_sft.py --config configs/post_training.yaml


## 7. DPO preference optimisation


In [ ]:
!python scripts/train_dpo.py --config configs/post_training.yaml --sft-adapter outputs/qwen3-financial-sft


## 8. Check artefacts


In [ ]:
from pathlib import Path
for path in [Path('outputs/qwen3-financial-sft'), Path('outputs/qwen3-financial-dpo')]:
    print(path, 'exists:', path.exists())
    if path.exists():
        for item in list(path.iterdir())[:10]:
            print('  ', item.name)
if USE_GOOGLE_DRIVE:
    !cp -r outputs/qwen3-financial-sft "$DRIVE_OUTPUT/"
    !cp -r outputs/qwen3-financial-dpo "$DRIVE_OUTPUT/"


## 9. Metric smoke check

The repository evaluation script currently scores prediction and gold text files; it does not yet generate model predictions from adapters. This cell validates the metric layer without fabricating model results.


In [ ]:
from finpost.evaluate import score

smoke = score(['1200', 'cloud services'], ['1200', 'Cloud Services'])
print(smoke)
assert smoke == {'numeric_em': 0.5, 'span_match': 1.0, 'combined': 0.75}


## 10. Compare base, SFT and DPO

Generate predictions from the base model, the SFT adapter and the DPO adapter on the same held-out validation prompts. Scores are logged as nested MLflow runs and printed as a comparison table.


In [ ]:
!python scripts/compare_models.py \
    --config configs/post_training.yaml \
    --sft-adapter outputs/qwen3-financial-sft \
    --dpo-adapter outputs/qwen3-financial-dpo \
    --limit 50


## 11. Iterate over combined scores

Every comparison is logged to MLflow as a nested run in the absolute sqlite store configured in step 4, so results survive Colab runtime restarts. Iterate over `metrics.combined` across experiments to tune LoRA rank, learning rate or DPO beta.


In [ ]:
import mlflow

mlflow.set_tracking_uri(config['mlflow']['tracking_uri'])

exps = mlflow.search_experiments()
print(f'Found {len(exps)} experiments in DB.')
for e in exps:
    print(f' - ID: {e.experiment_id}, Name: {e.name}')

runs = mlflow.search_runs(experiment_ids=[e.experiment_id for e in exps])
desired_cols = ['tags.mlflow.runName', 'metrics.combined', 'metrics.numeric_em', 'metrics.span_match']

if not runs.empty:
    existing_cols = [c for c in desired_cols if c in runs.columns]
    if 'metrics.combined' in existing_cols:
        df_display = runs[existing_cols].rename(columns={'tags.mlflow.runName': 'run_name'})
        print('\nComparison table:')
        print(df_display.dropna(subset=['metrics.combined']).sort_values('metrics.combined', ascending=False))
    else:
        print(f'\nRuns found, but missing metric columns. Available columns: {runs.columns.tolist()}')
else:
    print(f'\nNo runs found in the database at {mlflow.get_tracking_uri()}.')


## Full experiment notes

For a portfolio result, replace the smoke-run value with `max_samples: null`, use a suitable GPU, create higher-quality chosen/rejected preference pairs, and review the base/SFT/DPO comparison table. Record GPU type, run duration, peak VRAM, SFT loss, DPO loss and held-out metrics in MLflow.

Run notebook cells in order; the official Unsloth Colab guidance similarly recommends running cells sequentially or using Run all.
